In [78]:
import pymongo
from pymongo import MongoClient
import pandas as pd

try: 
    client = MongoClient("localhost", 5001)
    print("Connected successfully!!!") 
except:
    print("Could not connect to MongoDB")

project_id = "6543b8ee4180527babd20c3a" # Anna's Paper

data = client["dataset_db"]["fine_tuning"]

Connected successfully!!!


In [79]:
query = {}

cursor = data.find(query)

df = pd.DataFrame(list(cursor))

In [80]:
df

,_id,instruction,writing_intention,before_text,after_text
0,661d5d0794124061d4c41398,INSTRUCTION FOR WRITING INTENTION,Artifact,\section{Analysis on the Dreaddit dataset}\nTh...,% This must be in the first 5 lines to tell ar...
1,661d5d0894124061d4c41399,INSTRUCTION FOR WRITING INTENTION,Artifact,% This must be in the first 5 lines to tell ar...,\textit{Sentence: “I've been out of work for n...
2,661d5d0894124061d4c4139a,INSTRUCTION FOR WRITING INTENTION,Idea Generation,\textit{Sentence: “I've been out of work for n...,\textit{Sentence: “I've been out of work for n...
3,661d5d0894124061d4c4139b,INSTRUCTION FOR WRITING INTENTION,Artifact,\textit{Sentence: “I've been out of work for n...,% This must be in the first 5 lines to tell ar...
4,661d5d0894124061d4c4139c,INSTRUCTION FOR WRITING INTENTION,Artifact,% This must be in the first 5 lines to tell ar...,"Similar to almosthomeless, one example was giv..."
...,...,...,...,...,...
508,661f724bf89a58f89d73a7f6,INSTRUCTION FOR WRITING INTENTION,Semantic,% \preston{talk about the construction of the ...,% \preston{talk about the construction of the ...
509,661f724bf89a58f89d73a7f7,INSTRUCTION FOR WRITING INTENTION,Lexical,% \preston{talk about the construction of the ...,% \preston{talk about the construction of the ...
510,661f724bf89a58f89d73a7f8,INSTRUCTION FOR WRITING INTENTION,Drafting,% \preston{talk about the construction of the ...,% \preston{talk about the construction of the ...
511,661f724bf89a58f89d73a7f9,INSTRUCTION FOR WRITING INTENTION,Lexical,% \preston{talk about the construction of the ...,\section{Limitations}\label{sec:limitation}\nI...


In [87]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "mistralai/Mixtral-8x7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)

#model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

def tokenize(data):
  msg = [
      {"role": "user", "content": data}
  ]

  tokenized = tokenizer.apply_chat_template(msg, return_tensors="pt")
  #print("before size", before_tokenized.shape, "\nafter_size:", after_tokenized.shape)

  return tokenized.shape[1]

In [88]:
df["before_token_count"] = df["before_text"].map(tokenize)
df["after_token_count"] = df["after_text"].map(tokenize)

In [89]:
df

,_id,instruction,writing_intention,before_text,after_text,before_token_count,after_token_count
0,661d5d0794124061d4c41398,INSTRUCTION FOR WRITING INTENTION,Artifact,\section{Analysis on the Dreaddit dataset}\nTh...,% This must be in the first 5 lines to tell ar...,972,812
1,661d5d0894124061d4c41399,INSTRUCTION FOR WRITING INTENTION,Artifact,% This must be in the first 5 lines to tell ar...,\textit{Sentence: “I've been out of work for n...,812,645
2,661d5d0894124061d4c4139a,INSTRUCTION FOR WRITING INTENTION,Idea Generation,\textit{Sentence: “I've been out of work for n...,\textit{Sentence: “I've been out of work for n...,645,1556
3,661d5d0894124061d4c4139b,INSTRUCTION FOR WRITING INTENTION,Artifact,\textit{Sentence: “I've been out of work for n...,% This must be in the first 5 lines to tell ar...,1556,821
4,661d5d0894124061d4c4139c,INSTRUCTION FOR WRITING INTENTION,Artifact,% This must be in the first 5 lines to tell ar...,"Similar to almosthomeless, one example was giv...",821,1081
...,...,...,...,...,...,...,...
508,661f724bf89a58f89d73a7f6,INSTRUCTION FOR WRITING INTENTION,Semantic,% \preston{talk about the construction of the ...,% \preston{talk about the construction of the ...,2303,2292
509,661f724bf89a58f89d73a7f7,INSTRUCTION FOR WRITING INTENTION,Lexical,% \preston{talk about the construction of the ...,% \preston{talk about the construction of the ...,2292,2302
510,661f724bf89a58f89d73a7f8,INSTRUCTION FOR WRITING INTENTION,Drafting,% \preston{talk about the construction of the ...,% \preston{talk about the construction of the ...,2302,2359
511,661f724bf89a58f89d73a7f9,INSTRUCTION FOR WRITING INTENTION,Lexical,% \preston{talk about the construction of the ...,\section{Limitations}\label{sec:limitation}\nI...,2359,667


In [90]:
df["before_token_count"].describe()

count     513.000000
mean     1277.569201
std       525.091350
min         9.000000
25%       880.000000
50%      1279.000000
75%      1634.000000
max      2909.000000
Name: before_token_count, dtype: float64

In [ ]:

df["after_token_count"].describe()